# AccentSense: Deep Speech Training Pipeline (Google Colab GPU)
### Explainable Native Language Influence Detection in Indian English Speech
**Hardware Target**: Google Colab Free T4 GPU (Runtime -> Change runtime type -> T4 GPU)

This notebook trains `WavLM Base+` with **Attentive Statistics Pooling (ASP)** on speaker-disjoint splits across the 4 Regional Anchors:
1. `Northern_Hindi`
2. `Central_MP` (Malwa / Bhopal / Central Belt)
3. `Western_Gujarati`
4. `Southern_Tamil`

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install Required Dependencies
!pip install -q transformers datasets torchaudio librosa soundfile scikit-learn captum jiwer

In [ ]:
# 3. Clone Repository or Upload Files
import os
# If using GitHub repository:
# !git clone https://github.com/SameerGera/Accent-Sense.git
# %cd Accent-Sense

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("Environment initialized.")

In [ ]:
# 4. Launch WavLM Training on GPU
!python train_wavlm.py \
    --model_name "microsoft/wavlm-base-plus" \
    --epochs 15 \
    --batch_size 8 \
    --lr_head 1e-3 \
    --lr_backbone 5e-5 \
    --output_dir checkpoints

In [ ]:
# 5. Run Explainability & AUDC Faithfulness Benchmark
!python explain_speech.py \
    --checkpoint checkpoints/best_wavlm_accentsense.pt \
    --method integrated_gradients \
    --n_steps 15 \
    --benchmark_all

In [ ]:
# 6. Run Downstream Whisper ASR Benchmark
!python downstream_asr.py

In [ ]:
# 7. Download Trained Weights to Local Machine
from google.colab import files
if os.path.exists("checkpoints/best_wavlm_accentsense.pt"):
    print("Downloading trained checkpoint...")
    files.download("checkpoints/best_wavlm_accentsense.pt")
else:
    print("Checkpoint not found. Run training first.")